# RUN 04 — feature ablation and dimensionality reduction

Этот ноутбук делает дополнительный эксперимент для курсовой:

1. загружает датасет из CSV, как в обычном Jupyter-формате;
2. строит/использует target для next-day direction;
3. добавляет rolling/volume/market features;
4. сжимает FinBERT embeddings через PCA только на train;
5. сравнивает feature groups;
6. сохраняет `metrics.csv`, `confusion_matrix.csv`, `predictions.csv`, `dataset_summary.json`.

Главная логика: не подбирать модель ради accuracy, а проверить вклад групп признаков.

## 0. Импорты и подключение функций RUN 04

Ноутбук использует функции из `experiments/run_04_feature_ablation.py`, чтобы не дублировать большой код.

In [ ]:
from pathlib import Path
import sys
import json
import importlib.util

import numpy as np
import pandas as pd
from IPython.display import display

# Если ноутбук запущен из папки notebooks, поднимаемся в корень проекта.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RUN04_PATH = ROOT / 'experiments' / 'run_04_feature_ablation.py'
assert RUN04_PATH.exists(), f'Не найден файл: {RUN04_PATH}'

spec = importlib.util.spec_from_file_location('run04', RUN04_PATH)
run04 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run04)

print('ROOT:', ROOT)
print('RUN04:', RUN04_PATH)

## 1. Загрузка данных

Вариант 1: положи CSV в `data/primary_dataset.csv`.

Вариант 2: измени `DATA_PATH` вручную.

Вариант 3: если запускаешь в Colab, ноутбук предложит загрузить CSV через `files.upload()`.

In [ ]:
# Можно задать путь вручную:
DATA_PATH = None
# DATA_PATH = ROOT / 'data' / 'primary_dataset.csv'

candidate_paths = [
    ROOT / 'data' / 'primary_dataset.csv',
    ROOT / 'data' / 'run03_dataset.csv',
    ROOT / 'data' / 'model_dataset.csv',
    ROOT / 'data' / 'features.csv',
    ROOT / 'primary_dataset.csv',
]

if DATA_PATH is None:
    for p in candidate_paths:
        if p.exists():
            DATA_PATH = p
            break

if DATA_PATH is None:
    # Colab fallback. В обычном Jupyter этот блок просто пропустится.
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception:
        pass

if DATA_PATH is None:
    raise FileNotFoundError(
        'Не найден CSV. Положи файл в data/primary_dataset.csv или задай DATA_PATH вручную.'
    )

df_raw = pd.read_csv(DATA_PATH)
print('DATA_PATH:', DATA_PATH)
print('shape:', df_raw.shape)
display(df_raw.head())
display(pd.DataFrame({'column': df_raw.columns, 'dtype': df_raw.dtypes.astype(str)}).head(40))

## 2. Настройки эксперимента

Обычно можно оставить `None`: ноутбук сам попробует найти дату, target, close и embeddings.

Если auto-detect ошибся, пропиши названия колонок вручную.

In [ ]:
DATE_COL = None          # например: 'decision_date'
TARGET_COL = None        # например: 'y'
TICKER_COL = None        # если есть несколько тикеров, например: 'ticker'
EMBEDDING_PREFIX = None  # например: 'finbert_' или 'emb_'

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
PCA_COMPONENTS = (16, 32, 64)
C_GRID = (0.01, 0.1, 1.0, 10.0)
RANDOM_STATE = 42

OUTPUT_DIR = ROOT / 'artifacts' / 'run_04'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

## 3. Подготовка датасета

Здесь происходит:

- определение date column;
- построение `y`, если target не задан;
- добавление market/rolling/volume features;
- определение FinBERT embedding columns;
- хронологический train/validation/test split.

In [ ]:
df = df_raw.copy()
df.columns = [run04.normalize_name(c) for c in df.columns]

date_col = run04.detect_column(df, run04.DATE_CANDIDATES, DATE_COL)
if date_col is None:
    raise ValueError(f'Не удалось определить date column. Кандидаты: {run04.DATE_CANDIDATES}')

df = run04.coerce_dates(df, date_col)
df, target_col = run04.ensure_target(df, date_col, TARGET_COL, TICKER_COL)
df.attrs['date_col'] = date_col

df, market_groups = run04.make_market_features(df, date_col)

exclude_cols = [date_col, target_col]
if TICKER_COL:
    exclude_cols.append(TICKER_COL)

embedding_cols = run04.detect_embedding_columns(df, EMBEDDING_PREFIX, exclude_cols)
df, text_meta_cols = run04.add_text_meta_features(df, date_col, embedding_cols)

df = df.sort_values(date_col).reset_index(drop=True)
df.attrs['date_col'] = date_col
train_idx, val_idx, test_idx = run04.chronological_split(df, date_col, TRAIN_RATIO, VAL_RATIO)
all_idx = df.index.to_numpy()

print('date_col:', date_col)
print('target_col:', target_col)
print('n rows:', len(df))
print('n unique dates:', df[date_col].nunique())
print('date range:', df[date_col].min(), '—', df[date_col].max())
print('train / val / test:', len(train_idx), len(val_idx), len(test_idx))
print('embedding cols:', len(embedding_cols))
print('embedding sample:', embedding_cols[:10])

split_summary = pd.DataFrame([
    {'split': 'train', 'n': len(train_idx), 'date_min': df.iloc[train_idx][date_col].min(), 'date_max': df.iloc[train_idx][date_col].max(), 'pos': int(df.iloc[train_idx][target_col].sum())},
    {'split': 'validation', 'n': len(val_idx), 'date_min': df.iloc[val_idx][date_col].min(), 'date_max': df.iloc[val_idx][date_col].max(), 'pos': int(df.iloc[val_idx][target_col].sum())},
    {'split': 'test', 'n': len(test_idx), 'date_min': df.iloc[test_idx][date_col].min(), 'date_max': df.iloc[test_idx][date_col].max(), 'pos': int(df.iloc[test_idx][target_col].sum())},
])
display(split_summary)

## 4. Feature groups + PCA для FinBERT

PCA обучается только на train subset. Это важно: иначе будет preprocessing leakage.

In [ ]:
feature_groups = {
    'prices_raw': market_groups.get('prices_raw', []),
    'returns_rolling': market_groups.get('returns_rolling', []),
    'volume_rolling': market_groups.get('volume_rolling', []),
    'market_full': market_groups.get('market_full', []),
    'finbert_full': embedding_cols,
    'text_meta': text_meta_cols,
}
feature_groups['market_full_text_meta'] = list(dict.fromkeys(feature_groups['market_full'] + text_meta_cols))

pca_info = {}
for k in PCA_COMPONENTS:
    if embedding_cols:
        pca_df, pca_cols, info = run04.make_pca_features(
            df, embedding_cols, train_idx, all_idx, k, RANDOM_STATE
        )
        for col in pca_cols:
            df[col] = pca_df[col]
        pca_info[f'finbert_pca_{k}'] = info
        feature_groups[f'finbert_pca_{k}'] = pca_cols
        feature_groups[f'market_full_finbert_pca_{k}'] = list(dict.fromkeys(feature_groups['market_full'] + pca_cols))
        feature_groups[f'market_full_finbert_pca_{k}_text_meta'] = list(dict.fromkeys(feature_groups['market_full'] + pca_cols + text_meta_cols))

feature_sizes = pd.DataFrame([
    {'feature_group': name, 'n_features': len(cols)}
    for name, cols in feature_groups.items()
]).sort_values('n_features', ascending=False)
display(feature_sizes)
print(json.dumps(pca_info, indent=2, ensure_ascii=False))

## 5. Запуск baselines и Logistic Regression

Для Logistic Regression:

- `C` подбирается на validation;
- threshold подбирается на validation;
- test используется только один раз для финальной оценки.

In [ ]:
metrics_rows = []
cm_rows = []
prediction_frames = []
selection_rows = []

# Baseline 1: majority
m, cm, preds, details = run04.evaluate_majority(df, target_col, train_idx, test_idx)
metrics_rows.append({'model': 'baseline', 'feature_group': 'majority', **m, **details})
cm_rows.append({'model': 'baseline', 'feature_group': 'majority', **cm})
preds['model'] = 'baseline'
preds['feature_group'] = 'majority'
prediction_frames.append(preds)

# Baseline 2: previous direction
m, cm, preds, details = run04.evaluate_previous_direction(df, date_col, target_col, test_idx)
metrics_rows.append({'model': 'baseline', 'feature_group': 'previous_direction', **m, **details})
cm_rows.append({'model': 'baseline', 'feature_group': 'previous_direction', **cm})
preds['model'] = 'baseline'
preds['feature_group'] = 'previous_direction'
prediction_frames.append(preds)

# Logistic Regression over feature groups
for group_name, cols in feature_groups.items():
    cols = [c for c in cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if not cols:
        print('skip empty group:', group_name)
        continue
    print('run:', group_name, 'n_features=', len(cols))
    m, cm, preds, details = run04.fit_logreg_group(
        df=df,
        feature_cols=cols,
        target_col=target_col,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        c_grid=C_GRID,
        random_state=RANDOM_STATE,
    )
    metrics_rows.append({'model': 'LogisticRegression', 'feature_group': group_name, **m, **details})
    cm_rows.append({'model': 'LogisticRegression', 'feature_group': group_name, **cm})
    selection_rows.append({'feature_group': group_name, **details})
    preds['model'] = 'LogisticRegression'
    preds['feature_group'] = group_name
    prediction_frames.append(preds)

metrics = pd.DataFrame(metrics_rows).sort_values(['balanced_accuracy', 'accuracy'], ascending=False)
confusion = pd.DataFrame(cm_rows)
selections = pd.DataFrame(selection_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

display(metrics)
display(confusion)
display(selections.head(20))

## 6. Delta-анализ относительно `market_full`

Главный вопрос RUN 04: улучшают ли PCA-сжатые FinBERT-признаки расширенный market baseline?

In [ ]:
metrics_delta = metrics.copy()
market_rows = metrics_delta[metrics_delta['feature_group'] == 'market_full']
if len(market_rows) > 0:
    market = market_rows.iloc[0]
    for col in ['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']:
        metrics_delta[f'delta_vs_market_full_{col}'] = metrics_delta[col] - market[col]
else:
    print('market_full не найден, delta не посчитана')

interesting = metrics_delta[
    metrics_delta['feature_group'].str.contains('market_full|finbert_pca|finbert_full|majority|previous', regex=True)
].copy()
display(interesting.sort_values(['balanced_accuracy', 'accuracy'], ascending=False))

## 7. Сохранение результатов

После выполнения этой ячейки пришли мне:

- `artifacts/run_04/metrics.csv`;
- `artifacts/run_04/dataset_summary.json`;
- желательно `artifacts/run_04/validation_selection.csv`.

По ним я обновлю разделы 2.6, 2.7, 2.8 и заключение.

In [ ]:
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
confusion.to_csv(OUTPUT_DIR / 'confusion_matrix.csv', index=False)
selections.to_csv(OUTPUT_DIR / 'validation_selection.csv', index=False)
predictions.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)

feature_manifest = {name: list(map(str, cols)) for name, cols in feature_groups.items()}
with open(OUTPUT_DIR / 'feature_groups.json', 'w', encoding='utf-8') as f:
    json.dump(feature_manifest, f, ensure_ascii=False, indent=2, default=str)

summary = {
    'data_path': str(DATA_PATH),
    'date_col': date_col,
    'target_col': target_col,
    'n_observations': int(len(df)),
    'n_unique_dates': int(df[date_col].nunique()),
    'date_min': str(df[date_col].min().date()),
    'date_max': str(df[date_col].max().date()),
    'train_size': int(len(train_idx)),
    'validation_size': int(len(val_idx)),
    'test_size': int(len(test_idx)),
    'train_date_min': str(df.iloc[train_idx][date_col].min().date()),
    'train_date_max': str(df.iloc[train_idx][date_col].max().date()),
    'validation_date_min': str(df.iloc[val_idx][date_col].min().date()),
    'validation_date_max': str(df.iloc[val_idx][date_col].max().date()),
    'test_date_min': str(df.iloc[test_idx][date_col].min().date()),
    'test_date_max': str(df.iloc[test_idx][date_col].max().date()),
    'n_embedding_cols': int(len(embedding_cols)),
    'embedding_cols_sample': list(map(str, embedding_cols[:10])),
    'pca_info': pca_info,
    'feature_group_sizes': {name: len(cols) for name, cols in feature_groups.items()},
}
with open(OUTPUT_DIR / 'dataset_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print('saved to:', OUTPUT_DIR)
print('files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p)

## 8. LaTeX-таблица для курсовой

Эту таблицу можно вставить в раздел 2.8 после проверки смысла результатов.

In [ ]:
latex_cols = ['model', 'feature_group', 'accuracy', 'balanced_accuracy', 'f1', 'roc_auc']
latex_table = metrics[latex_cols].copy()
for c in ['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']:
    latex_table[c] = latex_table[c].round(3)

print(latex_table.to_latex(index=False, escape=False))